In [54]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
import glob
import os

In [55]:
df_pff = pd.read_csv('pffScoutingData.csv')
all_files = ['week' + str(i) +'.csv' for i in range(1,2)]

df_ngs = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

print(df_pff['pff_role'].unique())

['Pass' 'Pass Route' 'Pass Block' 'Pass Rush' 'Coverage']


In [56]:
merged_df = df_ngs.merge(df_pff[['gameId','playId','nflId','pff_role','pff_positionLinedUp']], left_on=['gameId', 'playId', 'nflId'], right_on=['gameId', 'playId','nflId'])

In [57]:
merged_df = merged_df[(merged_df['pff_role'] == 'Pass Block') | (merged_df['pff_role'] == 'Pass Rush')| (merged_df['pff_role'] == 'Pass')]

In [58]:
# N = 3
# vals = np.random.choice(merged_df[['label']].unique(), N, replace=False) 
# print (vals)
# ['C' 'A' 'B']

# df = df.set_index('label').loc[vals].reset_index()
merged_df

,gameId,playId,nflId,frameId,time,jerseyNumber,team,playDirection,x,y,s,a,dis,o,dir,event,pff_role,pff_positionLinedUp
0,2021090900,97,25511.0,1,2021-09-10T00:26:31.100,12.0,TB,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,None,Pass,QB
1,2021090900,97,25511.0,2,2021-09-10T00:26:31.200,12.0,TB,right,37.78,24.22,0.23,0.11,0.02,164.33,92.87,None,Pass,QB
2,2021090900,97,25511.0,3,2021-09-10T00:26:31.300,12.0,TB,right,37.78,24.24,0.16,0.10,0.01,160.24,68.55,None,Pass,QB
3,2021090900,97,25511.0,4,2021-09-10T00:26:31.400,12.0,TB,right,37.73,24.25,0.15,0.24,0.06,152.13,296.85,None,Pass,QB
4,2021090900,97,25511.0,5,2021-09-10T00:26:31.500,12.0,TB,right,37.69,24.26,0.25,0.18,0.04,148.33,287.55,None,Pass,QB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1069503,2021091300,4845,53460.0,30,2021-09-14T03:54:20.600,99.0,BAL,left,44.03,23.09,2.04,1.79,0.21,75.81,122.50,pass_forward,Pass Rush,LOLB
1069504,2021091300,4845,53460.0,31,2021-09-14T03:54:20.700,99.0,BAL,left,44.19,22.97,1.85,1.90,0.19,74.82,129.11,None,Pass Rush,LOLB
1069505,2021091300,4845,53460.0,32,2021-09-14T03:54:20.800,99.0,BAL,left,44.31,22.84,1.73,2.08,0.18,78.36,137.55,None,Pass Rush,LOLB
1069506,2021091300,4845,53460.0,33,2021-09-14T03:54:20.900,99.0,BAL,left,44.41,22.70,1.66,2.42,0.17,83.85,148.87,None,Pass Rush,LOLB


In [59]:
df_qb = merged_df[merged_df['pff_role'] == 'Pass'][['gameId','playId','frameId','time','nflId','x','y']]
df_qb

,gameId,playId,frameId,time,nflId,x,y
0,2021090900,97,1,2021-09-10T00:26:31.100,25511.0,37.77,24.22
1,2021090900,97,2,2021-09-10T00:26:31.200,25511.0,37.78,24.22
2,2021090900,97,3,2021-09-10T00:26:31.300,25511.0,37.78,24.24
3,2021090900,97,4,2021-09-10T00:26:31.400,25511.0,37.73,24.25
4,2021090900,97,5,2021-09-10T00:26:31.500,25511.0,37.69,24.26
...,...,...,...,...,...,...,...
1068891,2021091300,4845,30,2021-09-14T03:54:20.600,41265.0,52.49,24.69
1068892,2021091300,4845,31,2021-09-14T03:54:20.700,41265.0,52.77,24.71
1068893,2021091300,4845,32,2021-09-14T03:54:20.800,41265.0,53.03,24.71
1068894,2021091300,4845,33,2021-09-14T03:54:20.900,41265.0,53.30,24.68


In [60]:
df_pr = merged_df[merged_df['pff_role'] == 'Pass Rush'][['gameId','playId','frameId','time','nflId','x','y']]
df_pb = merged_df[merged_df['pff_role'] == 'Pass Block'][['gameId','playId','frameId','time','nflId','x','y']]


In [61]:
df_x = df_pr.merge(df_qb, left_on=['gameId', 'playId', 'frameId','time'], right_on=['gameId', 'playId','frameId','time'], suffixes=('_pr', '_qb')) 

In [62]:
df_test = df_x #df_x[(df_x['gameId'] ==2021090900) &   (df_x['playId'] ==97)&   (df_x['frameId'] <=10)]

In [63]:
df_final = df_test.merge(df_pb, left_on=['gameId', 'playId', 'frameId','time'], right_on=['gameId', 'playId','frameId','time'], suffixes=('' ,'_pb')) 

In [64]:
X = df_final[['x_pr','y_pr','x_qb','y_qb']].values

In [65]:
Y = df_final[['x','y']].values

In [66]:
print(X.shape, Y.shape)

(1114658, 4) (1114658, 2)
